# ⚖️ Week 3 hands-on: Model judges and human calibration

## Learning Objectives
In this notebook, you will learn:
1. **The judge prompt** - render rubric version 1 with the policy, the message and the reply
2. **A fail-closed parser** - accept valid judge responses and reject bad ones with a reason
3. **Agreement with human labels** - compute the confusion matrix, percent agreement and Cohen's kappa
4. **Pairwise position bias** - keep a winner only when both orders agree
5. **Breaking the judge workflow** - catch a fail-open parser and a copied review sheet

## Where this fits
This is the runnable companion to `week-03-judges-and-human-calibration-handson.md`, using the same code as `lab.py` (a test keeps the two identical). Read `week-03-judges-and-human-calibration-theory.md` first. Week 2 validated the dataset. This week grades v1's replies to the 30 Week 1 cases, all from the development split. Week 4 starts retrieval evaluation.

## Prerequisites
- `week-03-judges-and-human-calibration-theory.md`, and Weeks 1 and 2
- The course environment from `new_course/setup.md`, selected as this notebook's kernel
- No API keys for the offline parts. The optional live judge in Part 6 needs `openai` from `requirements-integrations.txt` and `OPENAI_API_KEY`, and costs money

**About the data:** the judge responses are hand-written mocks, not model output. The example review sheet is written by the course author, not human review. They exercise the code only.

---
## 📦 Step 1: Environment Setup

The setup cell finds `new_course/` and names every file this week reads.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Paths to the samples, prompt, policy and review sheets
# ============================================================================
import csv
import json
import os
import re
import sys
from collections import Counter
from pathlib import Path

# --- Find new_course/ from wherever Jupyter started (the notebook's folder or new_course/)
COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "support_agent.py").exists())
sys.path.insert(0, str(COURSE))

WEEK = COURSE / "weeks" / "week-03-judges-and-human-calibration"
FIXTURES = WEEK / "dataset"
SAMPLES = FIXTURES / "samples.jsonl"
PROMPT = WEEK / "handson" / "judge_prompt.md"
POLICY = COURSE / "datasets" / "policies" / "refund-policy-2026-01.md"
MOCK_RESPONSES = FIXTURES / "mock_judge_responses.jsonl"
EXAMPLE_LABELS = FIXTURES / "example_review_sheet.csv"
MOCK_PAIRWISE = FIXTURES / "mock_pairwise_responses.jsonl"

print(f"✅ Environment ready: {SAMPLES.relative_to(COURSE)}")

---
## 📄 Part 1: The Judge Prompt

The judge sees the policy, the customer message and the reply. It never sees the trace, the order owner or the expected outcome, so it grades the text alone. That makes the judge **reference-free**.

In [ ]:
# ============================================================================
# JUDGE PROMPT: Render rubric version 1 for case-0024
# ============================================================================
def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


def policy_text() -> str:
    """The policy without its front matter: what the judge and the reviewer read."""
    return POLICY.read_text().split("---", 2)[2].strip()


def policy_rule_ids(text: str) -> set[str]:
    return set(re.findall(r"^## (R\d+)\.", text, flags=re.MULTILINE))


def render_prompt(sample: dict, template: str | None = None) -> str:
    template = template or PROMPT.read_text()
    return (template.replace("{{policy}}", policy_text())
                    .replace("{{message}}", sample["message"])
                    .replace("{{reply}}", sample["final_text"]))


samples = read_jsonl(SAMPLES)
by_id = {s["case_id"]: s for s in samples}
prompt = render_prompt(by_id["case-0024"])
print(f"📄 {len(samples)} samples. Prompt for case-0024 ({len(prompt)} characters), last part:\n")
print(prompt[prompt.index("## Customer message"):])

---
## 🔍 Part 2: A Fail-Closed Parser

`parse_judgment` returns either a checked judgment or the reason it rejected the response. An unreadable or unsupported response is **invalid**, never a pass. The quote check rejects evidence that does not appear in the policy word for word.

In [ ]:
# ============================================================================
# PARSER: Validate a judge response
# ============================================================================
def parse_judgment(raw: str, policy: str) -> tuple[dict | None, str | None]:
    """Return (judgment, None) for a valid response or (None, reason) for an invalid one."""
    text = raw.strip()
    fenced = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.DOTALL)
    if fenced:  # models often wrap JSON in a Markdown code fence
        text = fenced.group(1)
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return None, "not JSON"
    if not isinstance(data, dict):
        return None, "not a JSON object"

    verdict, score = data.get("verdict"), data.get("score")
    rule_ids, quotes = data.get("rule_ids"), data.get("quotes")
    if verdict not in ("pass", "fail"):
        return None, f"verdict {verdict!r} is not pass or fail"
    if not isinstance(score, int) or isinstance(score, bool) or not 1 <= score <= 5:
        return None, f"score {score!r} is not an integer from 1 to 5"
    if (verdict == "pass") != (score >= 3):
        return None, f"verdict {verdict} contradicts score {score}"
    if not isinstance(rule_ids, list) or not rule_ids:
        return None, "no rule_ids cited"
    unknown = sorted(set(rule_ids) - policy_rule_ids(policy))
    if unknown:
        return None, f"cites rules that do not exist: {unknown}"
    if not isinstance(quotes, list) or not quotes:
        return None, "no quotes cited"
    flat_policy = " ".join(policy.split())
    for quote in quotes:
        if not isinstance(quote, str) or " ".join(quote.split()) not in flat_policy:
            return None, f"quote not found in policy: {quote!r}"
    return {"verdict": verdict, "score": score, "rule_ids": rule_ids,
            "quotes": quotes, "reasoning": str(data.get("reasoning", ""))}, None


policy = policy_text()
quote = "A customer can request a refund within 30 days of delivery."
attempts = {
    "valid": {"verdict": "pass", "score": 4, "rule_ids": ["R2"], "quotes": [quote], "reasoning": "ok"},
    "score out of range": {"verdict": "pass", "score": 7, "rule_ids": ["R2"], "quotes": [quote]},
    "contradiction": {"verdict": "pass", "score": 2, "rule_ids": ["R2"], "quotes": [quote]},
    "invented rule": {"verdict": "pass", "score": 4, "rule_ids": ["R9"], "quotes": [quote]},
    "invented quote": {"verdict": "pass", "score": 5, "rule_ids": ["R2"],
                       "quotes": ["Damaged items are always refundable."]},
}
for label, response in attempts.items():
    judgment, error = parse_judgment(json.dumps(response), policy)
    print(f"{'✅' if judgment else '❌'} {label:<20} {error or judgment['verdict']}")
print(f"❌ {'prose':<20} {parse_judgment('Verdict: pass', policy)[1]}")

---
## 🧮 Part 3: Agreement with Human Labels

**Cohen's kappa** corrects percent agreement for chance: (p_o − p_e) / (1 − p_e). The report also lists every disagreement and every run the judge passed although it made a forbidden tool call. Work out kappa by hand from the theory file's worked example before running the cell.

> **Key Insight**: The judge passes five replies that made a forbidden refund. Only the trace shows the harm.

In [ ]:
# ============================================================================
# AGREEMENT REPORT: Mock judge versus example labels
# ============================================================================
def forbidden_call(sample: dict) -> bool:
    """The Week 1 deterministic safety check, applied to a frozen sample."""
    return any(call["name"] in sample["forbidden_actions"] for call in sample["tool_calls"])


def load_labels(path: Path) -> dict[str, dict]:
    with path.open(newline="") as f:
        rows = list(csv.DictReader(f))
    labels = {}
    for row in rows:
        if not row["human_verdict"].strip():
            continue  # not reviewed yet
        if row["human_verdict"] not in ("pass", "fail"):
            raise ValueError(f"{row['case_id']}: human_verdict must be pass or fail")
        labels[row["case_id"]] = row
    return labels


def agreement(pairs: list[tuple[str, str]]) -> dict:
    """Compare (human, judge) verdict pairs. Positive class: "fail", a bad reply caught."""
    n = len(pairs)
    cells = Counter(pairs)
    tp, fp = cells[("fail", "fail")], cells[("pass", "fail")]
    fn, tn = cells[("fail", "pass")], cells[("pass", "pass")]
    observed = (tp + tn) / n if n else 0.0
    human_fail, judge_fail = (tp + fn) / n if n else 0, (tp + fp) / n if n else 0
    expected = human_fail * judge_fail + (1 - human_fail) * (1 - judge_fail)
    kappa = (observed - expected) / (1 - expected) if expected < 1 else 0.0
    return {"n": n, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "observed": observed, "expected": expected, "kappa": kappa}


def build_report(samples, responses, labels, policy) -> dict:
    judgments, invalid = {}, {}
    for r in responses:
        judgment, error = parse_judgment(r["raw"], policy)
        if judgment:
            judgments[r["case_id"]] = judgment
        else:
            invalid[r["case_id"]] = error
    reviewed = [s for s in samples if s["case_id"] in labels]
    pairs = [(labels[s["case_id"]]["human_verdict"], judgments[s["case_id"]]["verdict"])
             for s in reviewed if s["case_id"] in judgments]
    scores = [(int(labels[s["case_id"]]["human_score"]), judgments[s["case_id"]]["score"])
              for s in reviewed if s["case_id"] in judgments and labels[s["case_id"]]["human_score"]]
    return {
        "judgments": judgments,
        "invalid": invalid,
        "agreement": agreement(pairs),
        "score_exact": sum(h == j for h, j in scores) / len(scores) if scores else 0.0,
        "score_within_1": sum(abs(h - j) <= 1 for h, j in scores) / len(scores) if scores else 0.0,
        "disagreements": [s["case_id"] for s in reviewed if s["case_id"] in judgments
                          and labels[s["case_id"]]["human_verdict"] != judgments[s["case_id"]]["verdict"]],
        "label_sources": Counter(labels[s["case_id"]]["label_source"] for s in reviewed),
        "unreviewed": [s["case_id"] for s in samples if s["case_id"] not in labels],
        "judge_passed_forbidden": [s["case_id"] for s in samples if forbidden_call(s)
                                   and judgments.get(s["case_id"], {}).get("verdict") == "pass"],
    }


def print_report(report: dict, labels: dict) -> None:
    if not labels:
        print("No reviewed samples yet. Fill in human_verdict and human_score in the sheet first.")
        return
    sources = report["label_sources"]
    if set(sources) != {"human"}:
        print("LABEL STATUS: these labels are not all from human review "
              f"({dict(sources)}).\nThe agreement numbers below test the code. "
              "They are not a calibration result.\n")
    if report["unreviewed"]:
        print(f"Unreviewed samples: {len(report['unreviewed'])}\n")

    print(f"Judge responses: {len(report['judgments'])} valid, {len(report['invalid'])} invalid")
    for case_id, reason in sorted(report["invalid"].items()):
        print(f"  INVALID {case_id}: {reason}")

    a = report["agreement"]
    print(f"\nVerdicts compared on {a['n']} samples (positive = fail):")
    print(f"  both fail={a['tp']}  judge-only fail={a['fp']}  "
          f"human-only fail={a['fn']}  both pass={a['tn']}")
    print(f"  agreement={a['observed']:.3f}  chance agreement={a['expected']:.3f}  "
          f"Cohen's kappa={a['kappa']:.3f}")
    print(f"  scores: exact match {report['score_exact']:.0%}, within one point "
          f"{report['score_within_1']:.0%}")

    print("\nDisagreements:")
    for case_id in report["disagreements"]:
        j = report["judgments"][case_id]
        h = labels[case_id]
        print(f"  {case_id}  human={h['human_verdict']}/{h['human_score']}  "
              f"judge={j['verdict']}/{j['score']}  judge says: {j['reasoning']}")
        if h.get("notes"):
            print(f"  {'':<10}human note: {h['notes']}")

    print(f"\nJudge passed runs that made a forbidden tool call: {report['judge_passed_forbidden']}")


labels = load_labels(EXAMPLE_LABELS)
report = build_report(samples, read_jsonl(MOCK_RESPONSES), labels, policy)
print_report(report, labels)

To produce real evidence, label `dataset/human_review_sheet.csv` yourself without looking at the judge's grades, save it, and point `load_labels` at your copy in the cell below. The label-status warning disappears only when every reviewed row says `human`.

In [ ]:
# ============================================================================
# YOUR LABELS: Run the report on your own sheet
# ============================================================================
my_sheet = FIXTURES / "human_review_sheet.csv"   # change to your filled-in copy
my_labels = load_labels(my_sheet)
if my_labels:
    print_report(build_report(samples, read_jsonl(MOCK_RESPONSES), my_labels, policy), my_labels)
else:
    print(f"⚠️ {my_sheet.name} has no reviewed rows yet. Fill in human_verdict and human_score first.")

---
## ⚖️ Part 4: Pairwise Position Bias

Each pair is judged twice with the replies swapped. A winner counts only when both orders pick the same reply. Otherwise the preference followed the position, not the content.

In [ ]:
# ============================================================================
# PAIRWISE: Order-independent winners only
# ============================================================================
def pairwise_consistency(rows: list[dict]) -> dict[str, str]:
    """Each comparison is judged twice with the replies swapped. Keep a winner only if both agree."""
    by_pair: dict[str, dict[str, str]] = {}
    for row in rows:
        shown_first = row["order"][0]                          # "A" or "B"
        winner = shown_first if row["preferred"] == "first" else row["order"][1]
        by_pair.setdefault(row["pair_id"], {})[row["order"]] = winner
    result = {}
    for pair_id, verdicts in by_pair.items():
        winners = set(verdicts.values())
        result[pair_id] = winners.pop() if len(winners) == 1 else "position-dependent"
    return result


for pair_id, winner in pairwise_consistency(read_jsonl(MOCK_PAIRWISE)).items():
    print(f"{'⚠️' if winner == 'position-dependent' else '✅'} {pair_id}: {winner}")

---
## 🐛 Part 5: Breaking the Judge Workflow

### 5.1 🐛 Drill 1: a fail-open parser
A teammate makes unreadable responses count as a pass with score 3. The invalid count drops from 4 to 3 and case-0030 becomes a pass with no evidence.

In [ ]:
# ============================================================================
# DRILL 1: Count invalid responses with a fail-open parser
# ============================================================================
def parse_fail_open(raw: str, policy: str):
    judgment, error = parse_judgment(raw, policy)
    if error == "not JSON":  # the broken "fix"
        return {"verdict": "pass", "score": 3, "rule_ids": [], "quotes": [], "reasoning": ""}, None
    return judgment, error


responses = read_jsonl(MOCK_RESPONSES)
closed = sum(parse_judgment(r["raw"], policy)[0] is None for r in responses)
opened = sum(parse_fail_open(r["raw"], policy)[0] is None for r in responses)
print(f"❌ Invalid responses: fail-closed {closed}, fail-open {opened}")
print(f"⚠️ case-0030 under fail-open: {parse_fail_open(responses[29]['raw'], policy)[0]}")

### 5.2 🐛 Drill 2: perfect agreement from a copied sheet
These labels copy the judge's verdicts, as a reviewer with the judge's output on screen might. Agreement and kappa reach 1.000, and the 4 unreviewed samples are exactly the 4 invalid responses.

In [ ]:
# ============================================================================
# DRILL 2: Labels copied from the judge
# ============================================================================
copied = {}
for r in responses:
    judgment, _error = parse_judgment(r["raw"], policy)
    if judgment:
        copied[r["case_id"]] = {"human_verdict": judgment["verdict"], "human_score": str(judgment["score"]),
                                "label_source": "human", "notes": ""}
copied_report = build_report(samples, responses, copied, policy)
a = copied_report["agreement"]
print(f"⚠️ agreement={a['observed']:.3f} kappa={a['kappa']:.3f} unreviewed={copied_report['unreviewed']}")
print("✅ Relabel blind, and measure human-to-human agreement first.")

---
## 💳 Part 6: Optional Live Judge (paid)

This cell is tagged `skip-execution` because it calls a paid API: 30 requests, about 30,000 input tokens. Automated runs skip it. To run it by hand, install `requirements-integrations.txt`, set `OPENAI_API_KEY`, change `RUN_LIVE` to `True` and run the cell. It has been checked against `openai` 2.54.0 but not run against a model yet.

In [ ]:
# ============================================================================
# LIVE JUDGE: Grade every sample with a real model (costs money)
# ============================================================================
def judge_live(samples: list[dict], model: str, out: Path) -> None:
    """Call a real model once per sample. Costs money. Requires `openai` and OPENAI_API_KEY."""
    if not os.environ.get("OPENAI_API_KEY"):
        sys.exit("OPENAI_API_KEY is not set. The live judge is optional; the offline report does not need it.")
    from openai import OpenAI  # requirements-integrations.txt

    client = OpenAI()
    out.parent.mkdir(parents=True, exist_ok=True)
    with out.open("w") as f:
        for sample in samples:
            response = client.chat.completions.create(
                model=model,
                temperature=0,
                response_format={"type": "json_object"},
                messages=[{"role": "user", "content": render_prompt(sample)}],
            )
            raw = response.choices[0].message.content or ""
            f.write(json.dumps({"case_id": sample["case_id"], "raw": raw,
                                "source": f"live:{model}"}) + "\n")
            print(f"{sample['case_id']}: {raw[:80]}")
    print(f"wrote {out}")


RUN_LIVE = False  # set to True to spend money on 30 judge calls
if RUN_LIVE:
    live_out = WEEK / "handson" / "runs" / "live_judge_responses.jsonl"
    judge_live(samples, "gpt-4o-mini", live_out)
    print_report(build_report(samples, read_jsonl(live_out), labels, policy), labels)
else:
    print("⚠️ Live judge skipped. Set RUN_LIVE = True to run it.")

---
## 📝 Summary

In this notebook, we learned:

### 1. The judge prompt
- The judge grades text from the policy, message and reply. It cannot see the trace.

### 2. A fail-closed parser
- Malformed, contradictory or invented evidence makes a response invalid, and invalid responses are counted, never passed.

### 3. Agreement with human labels
- 85% agreement is kappa 0.505 once chance is removed. Read the disagreements: one judge error, two rubric problems.

### 4. Pairwise position bias
- Judge every pair in both orders. pair-0028 is position-dependent and counts as a tie.

### 5. Breaking the judge workflow
- A fail-open parser hides judge failures. Perfect agreement from labels copied from the judge is not evidence.

### Next Steps
- **`week-04-retrieval-evaluation-handson.ipynb`**: measures whether the help-centre search returns the right policy passage.